<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/dataset_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Precomputation and Loading for Titans

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

fatal: destination path 'kauldron' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog


fatal: destination path 'gemma' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'dialog' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from gemma import gm
import jax
import jax.numpy as jnp

original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)


from gemma.gm.nn import _gemma

model = _gemma.Gemma3_1B(
    dtype=jnp.bfloat16,
    # return_last_only=False,
    tokens="batch.tokens",
)

In [ ]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')
MAX_LENGTH = 512 # длина последовательность в токенах
max_new_tokens = 64
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - max_new_tokens) * 4
split_name = 'validation'

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [ ]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset(split_name):
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split=split_name, ##splits: 'test' | 'train' | 'validation'
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [ ]:
split_name = 'test'
def precompute_and_save():
    ds = get_source_dataset(split_name)
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / f"{split_name}.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей.")

precompute_and_save() # Раскомментируйте для запуска

Начинаю фильтрацию и сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test.array_record...


  1%|          | 142/17210 [00:00<00:12, 1403.44it/s]

Disabling pygrain multi-processing (unsupported in colab).


100%|██████████| 17210/17210 [00:12<00:00, 1372.10it/s]

Готово! Сохранено 6143 валидных записей.


## 3. Загрузка сохраненных данных

Эта функция заменяет ваш `get_train_dataset_tydi_qa` в основном цикле обучения.

In [ ]:
def get_precomputed_dataset(batch_size=16, tokenizer=None,  files=None):
    """Загружает и объединяет отфильтрованные данные из нескольких файлов.

    Args:
        batch_size: Размер батча.
        tokenizer: Токенизатор Gemma.
        files: Список имен файлов, например ['train.array_record', 'validation.array_record']
    """
    if files is None:
        files = [f'{split_name}.array_record']

    paths = [str(DATA_DIR / f) for f in files]
    print(f"Загрузка данных из: {paths}")

    # 1. Определяем источник данных (DataSource)
    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    # 2. Создаем Kauldron Dataset
    return kd.data.py.DataSource(
        data_source=data_source,
        shuffle=True,      # Перемешивание важно при объединении разных сплитов!
        num_epochs=None,   # Бесконечно для обучения
        batch_size=batch_size,
        transforms=[
            # Здесь применяются format_triviaqa и gm.data.Seq2SeqTask
            kd.data.py.Elements(keep=["tokens", "target", "loss_mask"]),
        ],
    )

## 4. Использование в kd.train.Trainer

Пример того, как это интегрируется в основной конфиг.

In [ ]:
'''
trainer = kd.train.Trainer(
    seed=42,
    workdir='./workdir',
    train_ds=get_precomputed_dataset(
        batch_size=BATCH_SIZE,
        tokenizer=tokenizer,
        max_length=MAX_CONTEXT_CHARS,
        files=['train.array_record', 'validation.array_record', 'test.array_record']
    ),
    model=model,
    # ... остальная конфигурация
)
'''

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [13]:
input_file = f"{split_name}.array_record"
output_file = f"{split_name}_gemma_generated.array_record"
input_path = os.path.join(str(DATA_DIR), input_file)
output_path = os.path.join(str(DATA_DIR), output_file)
reader = array_record.ArrayRecordReader(str(input_path))
writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

In [ ]:
writer.

In [14]:
reader.num_records()

6707

In [22]:
# ВАЖНО: Эту ячейку нужно запускать в окружении с загруженной моделью Gemma.

import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
from gemma import gm
from etils import epath
import os
import json

# 1. Функция форматирования промпта
def format_triviaqa_prompt(x):
    ctx = ""
    if x.get('search_results', {}).get('search_context', []):
        ctx = x['search_results']['search_context'][0]
    q = x["question"]
    return f"Context: {ctx}\n\nQuestion: {q}\n\nAnswer: "

# 2. Функция генерации датасета
def generate_and_save_offline_dataset(
    model,
    params,
    tokenizer,
    split_name,
    max_new_tokens=64
):
    input_file = f"{split_name}.array_record"
    output_file = f"{split_name}_gemma_generated.array_record"
    input_path = os.path.join(str(DATA_DIR), input_file)
    output_path = os.path.join(str(DATA_DIR), output_file)

    print(f"Чтение из {input_path}")
    print(f"Сохранение в {output_path}")

    # Инициализация семплера Gemma
    sampler = gm.text.Sampler(
        model=model,
        params=params,
        tokenizer=tokenizer,
    )

    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    # Чтение исходного ArrayRecord
    reader = array_record.ArrayRecordReader(str(input_path))

    num_records = reader.num_records()
    print(f"Всего записей: {num_records}")

    # ArrayRecordReader не итерируемый напрямую, используем range
    for i in tqdm.tqdm(range(10)):
        # В некоторых версиях read() без аргументов читает следующую запись
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)

        # # ОТЛАДКА: Печать структуры и содержания записи
        # print(f"\n--- Record {i} ---")
        # print(f"Full record: {record}")

        # Формируем prompt
        prompt = format_triviaqa_prompt(record)
        new_tokens = (2048 - len(prompt)) // 4
        if new_tokens > max_new_tokens:
            current_new_tokens = new_tokens
        else:
            current_new_tokens = max_new_tokens

        # print(f"Prompt: {prompt}")
        # Генерируем ответ (offline)
        response_text = sampler.sample(prompt, max_new_tokens=current_new_tokens)

        # Подменяем эталонный ответ сгенерированным
        if 'answer' not in record:
            record['answer'] = {}
        record['answer']['value'] = response_text

        # print(f"Gemma Answer: {record['answer']['value']}")

        # Записываем обратно
        writer.write(pickle.dumps(record))

        # Сброс на диск каждые 100 шагов
        if (i + 1) % 100 == 0:
            writer.close()
            writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    writer.close()
    print(f"Успешно сгенерировано и сохранено {num_records} записей.")

In [23]:
generate_and_save_offline_dataset(
    model=model,
    params=original_params,
    tokenizer= gm.text.Gemma3Tokenizer(),
    split_name='test',
    max_new_tokens=max_new_tokens

)

Чтение из /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test.array_record
Сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test_gemma_generated.array_record
Всего записей: 6143


  0%|          | 0/10 [00:00<?, ?it/s]


--- Record 0 ---
Full record: {'answer': {'aliases': [], 'matched_wiki_entity_name': b'<unk>', 'normalized_aliases': [], 'normalized_matched_wiki_entity_name': b'<unk>', 'normalized_value': b'<unk>', 'type': b'', 'value': b'<unk>'}, 'entity_pages': {'doc_source': [b'TagMe', b'TagMe'], 'filename': [b'Football_in_England.txt', b'English_Football_League.txt'], 'title': [b'Football in England', b'English Football League'], 'wiki_context': [b'Association football is the national sport in England, where the first modern set of rules for the code were established in 1863, which were a major influence on the development of the modern Laws of the Game. With over 40,000 association football clubs, England has more clubs involved in the code than any other country as well as the world\'s first club (Sheffield F.C.), the world\'s oldest professional association football club (Notts County F.C), the oldest national governing body (the Football Association), the first national team, the oldest nati

 10%|█         | 1/10 [00:04<00:37,  4.13s/it]

Gemma Answer: 1979
<end_of_turn>

--- Record 1 ---
Full record: {'answer': {'aliases': [], 'matched_wiki_entity_name': b'<unk>', 'normalized_aliases': [], 'normalized_matched_wiki_entity_name': b'<unk>', 'normalized_value': b'<unk>', 'type': b'', 'value': b'<unk>'}, 'entity_pages': {'doc_source': [b'TagMe', b'TagMe'], 'filename': [b'Toy.txt', b'Godtfred_Kirk_Christiansen.txt'], 'title': [b'Toy', b'Godtfred Kirk Christiansen'], 'wiki_context': [b'A toy is an item that can be used for play. Toys are generally played with by children and pets. Playing with toys is an enjoyable means of training young children for life in society. Different materials are used to make toys enjoyable to all ages. Many items are designed to serve as toys, but goods produced for other purposes can also be used. For instance, a small child may pick up a household item and "fly" it through the air as to pretend that it is an airplane. Another consideration is interactive digital entertainment. Some toys are prod

...

Gemma Answer: 8
Final Answer: The final answer is 8.
<end_of_turn>

--- Record 8 ---
Full record: {'answer': {'aliases': [], 'matched_wiki_entity_name': b'<unk>', 'normalized_aliases': [], 'normalized_matched_wiki_entity_name': b'<unk>', 'normalized_value': b'<unk>', 'type': b'', 'value': b'<unk>'}, 'entity_pages': {'doc_source': [b'TagMe'], 'filename': [b'Bamboo.txt'], 'title': [b'Bamboo'], 'wiki_context': [b'The bamboos  are a subfamily (Bambusoideae) of flowering perennial evergreen plants in the grass family Poaceae.\n\nGiant bamboos are the largest members of the grass family. In bamboo, the internodal regions of the stem are usually hollow and the vascular bundles in the cross section are scattered throughout the stem instead of in a cylindrical arrangement. The dicotyledonous woody xylem is also absent. The absence of secondary growth wood causes the stems of monocots, including the palms and large bamboos, to be columnar rather than tapering.Botany; Wilson,C.L. and Loomis,W.E. 

 90%|█████████ | 9/10 [00:25<00:02,  2.69s/it]

Gemma Answer: 

Kendo

The Japanese art of fencing which uses bamboo swords is Kendo.

---

**Explanation:**

The text explicitly states: "One of Japan's martial arts employing techniques of fencing based on the two-handed sword of the samurai, kendo is now a system of mental and physical training practised

--- Record 9 ---
Full record: {'answer': {'aliases': [], 'matched_wiki_entity_name': b'<unk>', 'normalized_aliases': [], 'normalized_matched_wiki_entity_name': b'<unk>', 'normalized_value': b'<unk>', 'type': b'', 'value': b'<unk>'}, 'entity_pages': {'doc_source': [b'TagMe'], 'filename': [b'New_Miami_Stadium.txt'], 'title': [b'New Miami Stadium'], 'wiki_context': [b'New Miami Stadium is a multi-purpose stadium located in Miami Gardens, Florida, a suburb north of Miami. It is the home stadium of the National Football League (NFL)\'s Miami Dolphins, and the University of Miami Hurricanes college football team. The facility also hosts the Orange Bowl, an annual college football bowl ga

100%|██████████| 10/10 [00:29<00:00,  2.96s/it]

Gemma Answer: 

The answer is: Miami Marlins.

Explanation: The text states that "The stadium has also been Pro Player Stadium, Dolphins Stadium and Landshark Stadium." It then adds that "On the club level of the stadium is the Gallery of Legends, which includes a history of the team, classic photos and the players
Успешно сгенерировано и сохранено 6143 записей.


In [24]:
import array_record.python.array_record_module as array_record
import pickle
import os

# Путь к сгенерированному файлу
generated_path = os.path.join(str(DATA_DIR), 'test_gemma_generated.array_record')

if os.path.exists(generated_path):
    reader = array_record.ArrayRecordReader(generated_path)
    num_records = reader.num_records()
    print(f"Всего записей в файле: {num_records}")

    # Читаем первые 3 записи для проверки
    for i in range(min(3, num_records)):
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)
        print(f"\n--- Проверка записи {i} ---")
        print(f"Question: {record.get('question', 'N/A')}")
        print(f"Gemma Answer (saved): {record.get('answer', {}).get('value', 'N/A')}")

    reader.close()
else:
    print(f"Файл не найден: {generated_path}")

Всего записей в файле: 10

--- Проверка записи 0 ---
Question: b'In 1979 , which player in the English football league became the first to be transferred for \xc2\xa31 million ?'
Gemma Answer (saved): 1979
<end_of_turn>

--- Проверка записи 1 ---
Question: b'What toy did Godtfred Kirk Christiansen invent in 1955?'
Gemma Answer (saved): 1955 | Brickipedia | Fandom powered by Wikia
The text states that Godtfred Kirk Christiansen demonstrated the LEGO bricks at a toy fair in Nuremberg, Germany, but judges did not find them very appealing.
The text does not provide any information about what he invented.
Therefore, the

--- Проверка записи 2 ---
Question: b'Bokken (or bokuto), katana, wakizashi, and shinai are types of what, used in Japanese martial arts?'
Gemma Answer (saved): 1.  Kendo practice sword
2.  Katana swords
3.  Shinai
4.  Wakizashi
5.  Bokken

The correct answer is 3.

Let's break down why:

*   The text explicitly states that Bokken (b'B
